In [13]:
import pandas as pd
from shared_modeling import load_or_create_master_split_ids, run_model_experiment

In [14]:
predictors = ['oDM', 'acog_PEgHTN', 'ChronHTN'] # diabetes, preeclampsia, chronic hypertension
df_health = pd.read_csv('../Data/PREGNANCY_OUTCOMES.csv', usecols=predictors + ['PublicID'])
df_outcomes = pd.read_csv('../Data/Modified/Outcome.csv', usecols=['PublicID', 'MH_outcome'])

# Create the master split once and persist it for reuse in other notebooks.
split_path = 'master_split_ids.csv'
train_ids, test_ids = load_or_create_master_split_ids(df_outcomes, split_path)
df_outcomes

,PublicID,MH_outcome
0,00004O,1
1,00007I,1
2,00008G,0
3,00015J,0
4,00016H,1
...,...,...
7736,17349I,1
7737,17350A,1
7738,17351V,0
7739,17352T,1


In [17]:

df = pd.merge(df_health, df_outcomes, on='PublicID', how='inner')
df


,PublicID,oDM,ChronHTN,acog_PEgHTN,MH_outcome
0,00004O,3.0,2.0,7.0,1
1,00007I,3.0,2.0,7.0,1
2,00008G,3.0,2.0,7.0,0
3,00015J,3.0,2.0,7.0,0
4,00016H,3.0,2.0,7.0,1
...,...,...,...,...,...
7736,17349I,3.0,2.0,7.0,1
7737,17350A,3.0,2.0,3.0,1
7738,17351V,3.0,2.0,7.0,0
7739,17352T,3.0,2.0,7.0,1


In [18]:
X = df.drop(['MH_outcome', 'PublicID'], axis=1)
y = df['MH_outcome']

train_df = df[df['PublicID'].isin(train_ids)].copy()
test_df = df[df['PublicID'].isin(test_ids)].copy()

X_train = train_df.drop(['MH_outcome', 'PublicID'], axis=1)
X_test = test_df.drop(['MH_outcome', 'PublicID'], axis=1)
y_train = train_df['MH_outcome']
y_test = test_df['MH_outcome']

y.value_counts()


MH_outcome
0    4591
1    3150
Name: count, dtype: int64

In [19]:
X

,oDM,ChronHTN,acog_PEgHTN
0,3.0,2.0,7.0
1,3.0,2.0,7.0
2,3.0,2.0,7.0
3,3.0,2.0,7.0
4,3.0,2.0,7.0
...,...,...,...
7736,3.0,2.0,7.0
7737,3.0,2.0,3.0
7738,3.0,2.0,7.0
7739,3.0,2.0,7.0


In [21]:
# Run the LR pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'lr',
    predictors
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters found: {'classifier__C': 0.1, 'classifier__l1_ratio': 0.0}
Best Macro F1 Score: 0.4933
Model Coefficients:
num__oDM: -0.0025343675662081095
num__acog_PEgHTN: -0.03994653174967902
num__ChronHTN: -0.08346991065592847
Evaluation Metrics for LR with shared preprocessing and macro F1 grid search:
Accuracy: 0.5468
Precision: 0.5019
Recall: 0.5015
F1-score: 0.4890
ROC AUC: 0.5069


In [22]:
# Run the RF pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'rf',
    predictors
)

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters found: {'classifier__max_depth': 20, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 700}
Best Macro F1 Score: 0.4758
Feature Importances:
num__oDM: 0.24265832199010265
num__acog_PEgHTN: 0.4698224864149897
num__ChronHTN: 0.2875191915949077
Evaluation Metrics for RF with shared preprocessing and macro F1 grid search:
Accuracy: 0.5617
Precision: 0.5091
Recall: 0.5060
F1-score: 0.4818
ROC AUC: 0.5057


In [23]:
# Run the XGBoost pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'xgb',
    predictors
)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found: {'classifier__colsample_bytree': 1.0, 'classifier__learning_rate': 0.001, 'classifier__max_depth': 4, 'classifier__n_estimators': 100, 'classifier__subsample': 0.8}
Best Macro F1 Score: 0.4816
Feature Importances:
num__oDM: 0.09602860361337662
num__acog_PEgHTN: 0.14812393486499786
num__ChronHTN: 0.7558474540710449
Evaluation Metrics for XGB with shared preprocessing and macro F1 grid search:
Accuracy: 0.5591
Precision: 0.5040
Recall: 0.5026
F1-score: 0.4769
ROC AUC: 0.4995


In [24]:
# Run the SVM pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'svm',
    predictors
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters found: {'classifier__estimator__C': 1, 'classifier__estimator__gamma': 0.01, 'classifier__estimator__kernel': 'rbf'}
Best Macro F1 Score: 0.5043
Evaluation Metrics for SVM with shared preprocessing and macro F1 grid search:
Accuracy: 0.5449
Precision: 0.5072
Recall: 0.5061
F1-score: 0.4991
ROC AUC: 0.5063
